# Market Risk Dashboard
## VaR, Expected Shortfall, GARCH and Machine Learning

**Author:** Yanis Ait Ouamara  
**Program:** Master 2 Finance Technology Data — University Paris 1 Panthéon-Sorbonne  
**Period:** 2015-2024  

### Portfolio
| Ticker | Company | Sector | Geography |
|--------|---------|--------|-----------|
| AAPL | Apple | Technology | USA |
| JPM | JPMorgan | Banking | USA |
| MC.PA | LVMH | Luxury | France |
| NESN.SW | Nestle | Consumer Staples | Switzerland |
| TTE.PA | TotalEnergies | Energy | France |

### Objective
This project builds a comprehensive market risk framework combining classical risk management methods (VaR, Expected Shortfall, GARCH) with modern machine learning techniques (XGBoost, SHAP) to measure, predict and explain market risk on a diversified international portfolio.

## Step 1 — Data Collection

We download historical closing prices for our 5 assets using the yfinance library, covering the period from January 2015 to December 2024. This gives us approximately 10 years of daily data, including major market events such as the Covid crash (2020) and the Ukraine war shock (2022).

In [ ]:
# Install required libraries if needed
# !pip install yfinance arch xgboost shap

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Define tickers and time period
tickers = ['AAPL', 'JPM', 'MC.PA', 'NESN.SW', 'TTE.PA']
start_date = '2015-01-01'
end_date = '2024-12-31'

# Download closing prices
prices = yf.download(tickers, start=start_date, end=end_date)['Close']

print(f'Dataset dimensions: {prices.shape}')
print('\nFirst rows:')
print(prices.head())
print('\nMissing values per asset:')
print(prices.isnull().sum())

## Step 2 — Data Cleaning and Log Returns

Missing values arise from differences in trading calendars between the US, France and Switzerland. We use forward-fill to replace them with the last known price. We then compute daily log-returns, which are stationary by construction — a necessary condition for all subsequent models.

In [ ]:
# Forward fill missing values
prices_clean = prices.ffill()

print('Missing values after cleaning:')
print(prices_clean.isnull().sum())

# Compute daily log-returns: r_t = ln(P_t / P_t-1)
returns = np.log(prices_clean / prices_clean.shift(1))
returns = returns.dropna()

print(f'\nFinal dimensions: {returns.shape}')
print('\nDescriptive statistics:')
print(returns.describe())

## Step 3 — Visualizing Returns

Before computing any risk measure, we visualize the return series for each asset. The key pattern to observe is volatility clustering — periods of high volatility tend to be followed by more high volatility, and vice versa. This is the fundamental motivation for GARCH models later in the project.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 15))

for i, ticker in enumerate(returns.columns):
    axes[i].plot(returns.index, returns[ticker],
                linewidth=0.5, color='steelblue')
    axes[i].set_title(f'{ticker} — Daily Log Returns')
    axes[i].set_ylabel('Return')
    axes[i].axhline(y=0, color='red',
                   linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.savefig('returns_plot.png', dpi=150)
plt.show()

print('Volatility clustering is clearly visible:')
print('- Covid crash: March 2020')
print('- Ukraine war: February 2022')
print('- Calm periods: 2017-2019, 2023')

## Step 4 — Parametric VaR

The parametric VaR assumes returns follow a normal distribution with constant mean and volatility. It is the simplest method and serves as a benchmark. The formula is: VaR = mu + z * sigma, where z is the quantile of the standard normal distribution at the chosen confidence level.

The key limitation of this approach is that it assumes constant volatility — the same sigma is used for a calm day in 2017 and a crisis day in March 2020.

In [ ]:
confidence_level = 0.95
alpha = 1 - confidence_level

var_parametric = {}

for ticker in returns.columns:
    mu = returns[ticker].mean()
    sigma = returns[ticker].std()
    z = norm.ppf(alpha)
    var = mu + z * sigma
    var_parametric[ticker] = var

print('Parametric VaR at 95% confidence level:')
for ticker, var in var_parametric.items():
    print(f'{ticker}: {var*100:.2f}%')

print('\nInterpretation: on 95% of trading days, the loss does not exceed the VaR threshold.')
print('Nestle is the least risky asset (-1.65%), consistent with its defensive profile.')
print('Limitation: assumes constant volatility — clearly violated in financial markets.')

## Step 5 — Historical VaR

The historical VaR makes no assumption about the return distribution. It simply takes the empirical quantile of observed returns at the chosen confidence level. This approach is more realistic but depends entirely on the historical sample — it implicitly assumes the past is representative of the future.

In [ ]:
var_historical = {}

for ticker in returns.columns:
    var = returns[ticker].quantile(alpha)
    var_historical[ticker] = var

print('Comparison — Parametric vs Historical VaR at 95%:')
print(f"{'Ticker':<10} {'Parametric':>12} {'Historical':>12}")
print('-' * 36)
for ticker in returns.columns:
    vp = var_parametric[ticker]
    vh = var_historical[ticker]
    print(f'{ticker:<10} {vp*100:>11.2f}% {vh*100:>11.2f}%')

print('\nObservation: parametric VaR slightly overestimates risk compared to historical VaR.')
print('This suggests the 2015-2024 period was relatively benign despite the Covid shock.')

## Step 6 — Monte Carlo VaR

Monte Carlo VaR simulates a large number of return scenarios and takes the empirical quantile. With a normal distribution assumption, it should produce results close to the parametric VaR. The real power of Monte Carlo appears when simulating non-normal distributions with fat tails or jumps.

In [ ]:
np.random.seed(42)
n_simulations = 10000

var_montecarlo = {}

for ticker in returns.columns:
    mu = returns[ticker].mean()
    sigma = returns[ticker].std()
    simulated_returns = np.random.normal(mu, sigma, n_simulations)
    var = np.percentile(simulated_returns, 5)
    var_montecarlo[ticker] = var

print('Comparison of 3 VaR methods at 95%:')
print(f"{'Ticker':<10} {'Parametric':>12} {'Historical':>12} {'Monte Carlo':>13}")
print('-' * 50)
for ticker in returns.columns:
    vp = var_parametric[ticker]
    vh = var_historical[ticker]
    vm = var_montecarlo[ticker]
    print(f'{ticker:<10} {vp*100:>11.2f}% {vh*100:>11.2f}% {vm*100:>12.2f}%')

print('\nObservation: Monte Carlo and Parametric VaR are very close.')
print('Both rely on the same assumptions — normal distribution and constant sigma.')
print('With 10,000 simulations, they naturally converge to the same result.')

## Step 7 — Expected Shortfall (CVaR)

The VaR tells us the loss threshold at a given confidence level, but says nothing about how bad losses can be beyond that threshold. The Expected Shortfall (also called CVaR) fills this gap by computing the average loss in the worst 5% of scenarios. Basel III adopted ES as the standard regulatory risk measure precisely because of this property.

In [ ]:
es_historical = {}

for ticker in returns.columns:
    var = var_historical[ticker]
    tail_returns = returns[ticker][returns[ticker] <= var]
    es = tail_returns.mean()
    es_historical[ticker] = es

print('Comparison — Historical VaR vs Expected Shortfall at 95%:')
print(f"{'Ticker':<10} {'VaR 95%':>10} {'ES 95%':>10} {'Ratio ES/VaR':>14}")
print('-' * 48)
for ticker in returns.columns:
    var = var_historical[ticker]
    es = es_historical[ticker]
    ratio = es / var
    print(f'{ticker:<10} {var*100:>9.2f}% {es*100:>9.2f}% {ratio:>13.2f}x')

print('\nKey finding: ES averages 1.5x the VaR across all assets.')
print('When the VaR threshold is breached, losses are on average 50% worse.')
print('This is why Basel III replaced VaR with ES as the regulatory standard.')

## Step 8 — GARCH(1,1) Dynamic Volatility

All three VaR methods above assume constant volatility. GARCH (Generalized Autoregressive Conditional Heteroskedasticity) models time-varying volatility by making today's variance a function of both past squared residuals (the ARCH term) and past variance (the GARCH term). This allows the VaR to adapt in real time to changing market conditions.

In [ ]:
from arch import arch_model

garch_volatility = {}
var_garch = {}

for ticker in returns.columns:
    # Scale returns to percentage for numerical stability
    r = returns[ticker] * 100
    
    # Fit GARCH(1,1) model
    model = arch_model(r, vol='Garch', p=1, q=1, dist='normal')
    result = model.fit(disp='off')
    
    # Extract conditional volatility and rescale
    cond_vol = result.conditional_volatility / 100
    garch_volatility[ticker] = cond_vol
    
    # Compute dynamic VaR using GARCH volatility
    mu = returns[ticker].mean()
    z = norm.ppf(alpha)
    var_dynamic = mu + z * cond_vol
    var_garch[ticker] = var_dynamic
    
    alpha_param = result.params['alpha[1]']
    beta_param = result.params['beta[1]']
    print(f"{ticker} — omega: {result.params['omega']:.6f}, "
          f"alpha: {alpha_param:.4f}, "
          f"beta: {beta_param:.4f}, "
          f"alpha+beta: {alpha_param+beta_param:.4f}")

print('\nAll models are stationary — alpha + beta < 1 for all assets.')
print('TotalEnergies shows the highest persistence (0.99), consistent with oil price exposure.')
print('JPMorgan shows the highest sensitivity to new shocks (alpha = 0.12).')

In [ ]:
# Visualize dynamic VaR against actual returns
fig, axes = plt.subplots(5, 1, figsize=(12, 15))

for i, ticker in enumerate(returns.columns):
    axes[i].plot(returns.index, returns[ticker]*100,
                color='grey', linewidth=0.5, label='Returns', alpha=0.7)
    axes[i].plot(var_garch[ticker].index, var_garch[ticker]*100,
                color='red', linewidth=1, label='GARCH VaR 95%')
    axes[i].set_title(f'{ticker} — Dynamic GARCH VaR')
    axes[i].set_ylabel('Return (%)')
    axes[i].legend(loc='lower right')

plt.tight_layout()
plt.savefig('var_garch_plot.png', dpi=150)
plt.show()

print('The GARCH VaR widens during crises (Covid 2020, Ukraine 2022)')
print('and narrows during calm periods (2017-2019, 2023-2024).')
print('This is the key advantage over static VaR methods.')

In [ ]:
# Final comparison of all 4 VaR methods
print('Comparison of all 4 VaR methods at 95%:')
print(f"{'Ticker':<10} {'Param':>8} {'Hist':>8} {'MC':>8} {'GARCH avg':>12}")
print('-' * 52)

for ticker in returns.columns:
    vp = var_parametric[ticker]
    vh = var_historical[ticker]
    vm = var_montecarlo[ticker]
    vg = var_garch[ticker].mean()
    print(f'{ticker:<10} {vp*100:>7.2f}% {vh*100:>7.2f}% {vm*100:>7.2f}% {vg*100:>11.2f}%')

print('\nNote: GARCH average is lower because calm periods dominate the sample.')
print('The real advantage of GARCH is its real-time adaptation, not its average value.')

## Step 9 — Machine Learning: Stress Day Prediction

We now move beyond measuring risk to predicting it. We train a binary classifier to predict whether tomorrow will be a stress day — defined as a day where the equally-weighted portfolio return falls below its historical VaR. This is a practical early warning system for risk managers.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb

# Build equally weighted portfolio returns
portfolio_returns = returns.mean(axis=1)

# Define the VaR threshold for the portfolio
var_portfolio = portfolio_returns.quantile(0.05)
print(f'Portfolio VaR at 95%: {var_portfolio*100:.2f}%')

# Binary target: 1 if stress day, 0 if normal day
y = (portfolio_returns < var_portfolio).astype(int)
print(f'\nClass distribution:')
print(y.value_counts())
print(f'Proportion of stress days: {y.mean()*100:.2f}%')
print('\nNote: 95/5 imbalance requires class reweighting in the models.')

In [ ]:
# Build the feature matrix
features = pd.DataFrame(index=returns.index)

# Memory of past returns
features['portfolio_return_lag1'] = portfolio_returns.shift(1)
features['portfolio_return_lag2'] = portfolio_returns.shift(2)
features['portfolio_return_lag3'] = portfolio_returns.shift(3)

# Short and medium term realized volatility
features['realized_vol_5d'] = portfolio_returns.rolling(5).std()
features['realized_vol_20d'] = portfolio_returns.rolling(20).std()

# GARCH conditional volatility as a feature
portfolio_garch = pd.DataFrame(garch_volatility).mean(axis=1)
features['garch_vol'] = portfolio_garch.values

# Cumulative returns over different horizons
features['return_5d'] = portfolio_returns.rolling(5).sum()
features['return_20d'] = portfolio_returns.rolling(20).sum()

# Align features and target, drop NaN
df_ml = features.copy()
df_ml['target'] = y
df_ml = df_ml.dropna()

X = df_ml.drop('target', axis=1)
y_clean = df_ml['target']

print(f'ML dataset: {X.shape[0]} observations, {X.shape[1]} features')
print(f'Stress days: {y_clean.sum()}')
print(f'Features: {X.columns.tolist()}')

In [ ]:
# Time series split — no shuffling to respect temporal order
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clean, test_size=0.2, shuffle=False)

print(f'Training set: {X_train.shape[0]} observations')
print(f'Test set: {X_test.shape[0]} observations')
print(f'Stress days in test: {y_test.sum()}')

# Random Forest with class rebalancing
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# XGBoost with scale_pos_weight to handle imbalance
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
    random_state=42,
    eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print('\nRandom Forest results:')
print(classification_report(y_test, y_pred_rf))
print(f'AUC-ROC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}')

print('\nXGBoost results:')
print(classification_report(y_test, y_pred_xgb))
print(f'AUC-ROC: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]):.4f}')

In [ ]:
# Visualize stress probabilities over time
prob_stress_xgb = xgb_model.predict_proba(X_test)[:,1]
prob_stress_rf = rf.predict_proba(X_test)[:,1]

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(X_test.index, prob_stress_xgb,
            color='red', linewidth=0.8, label='Stress probability')
axes[0].axhline(y=0.5, color='black', linestyle='--', linewidth=0.5)
axes[0].scatter(X_test.index[y_test==1], prob_stress_xgb[y_test==1],
               color='darkred', s=50, zorder=5, label='Actual stress days')
axes[0].set_title('XGBoost — Stress Day Probability')
axes[0].set_ylabel('Probability')
axes[0].legend()

axes[1].plot(X_test.index, prob_stress_rf,
            color='steelblue', linewidth=0.8, label='Stress probability')
axes[1].axhline(y=0.5, color='black', linestyle='--', linewidth=0.5)
axes[1].scatter(X_test.index[y_test==1], prob_stress_rf[y_test==1],
               color='darkred', s=50, zorder=5, label='Actual stress days')
axes[1].set_title('Random Forest — Stress Day Probability')
axes[1].set_ylabel('Probability')
axes[1].legend()

plt.tight_layout()
plt.savefig('stress_probability.png', dpi=150)
plt.show()

print('XGBoost clearly outperforms Random Forest.')
print('Actual stress days correspond to probability spikes in XGBoost.')
print('Random Forest never crosses the 0.5 threshold despite a high AUC-ROC.')

## Step 10 — SHAP Explainability

XGBoost predicts stress days well, but in a risk management context it is not enough to know what the model predicts — we need to understand why. SHAP (SHapley Additive Explanations) decomposes each prediction into the individual contribution of each feature. This is essential for regulatory compliance and model validation.

In [ ]:
import shap

# Compute SHAP values for XGBoost
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Global feature importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test,
                 plot_type='bar', show=False)
plt.title('SHAP Feature Importance — XGBoost')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150)
plt.show()

# Beeswarm plot — direction of impact
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Beeswarm Plot — XGBoost')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150)
plt.show()

print('Top predictors of stress days:')
print('1. return_5d — 5-day cumulative return (most important by far)')
print('2. realized_vol_5d — short-term realized volatility')
print('3. realized_vol_20d — medium-term realized volatility')
print('\nInterpretation: recent negative returns combined with high short-term')
print('volatility are the strongest early warning signals for stress days.')

## Step 11 — Backtesting: Kupiec and Christoffersen Tests

A VaR model is only useful if it is well calibrated. Backtesting verifies this by comparing predicted risk thresholds against actual observed losses. We use two complementary tests:

- **Kupiec test**: checks whether the frequency of VaR breaches matches the expected level (5% for a 95% VaR)
- **Christoffersen test**: checks whether breaches are independent — not clustered in crisis periods

In [ ]:
def kupiec_test(returns, var, confidence=0.95):
    alpha = 1 - confidence
    exceptions = (returns < var).astype(int)
    n = len(exceptions)
    x = exceptions.sum()
    p_hat = x / n
    if p_hat == 0 or p_hat == 1:
        return None, None, x, p_hat
    lr = -2 * np.log(
        ((1-alpha)**(n-x) * alpha**x) /
        ((1-p_hat)**(n-x) * p_hat**x))
    p_value = 1 - stats.chi2.cdf(lr, df=1)
    return lr, p_value, x, p_hat


def christoffersen_test(returns, var):
    exceptions = (returns < var).astype(int).values
    n00, n01, n10, n11 = 0, 0, 0, 0
    for i in range(1, len(exceptions)):
        if exceptions[i-1] == 0 and exceptions[i] == 0:
            n00 += 1
        elif exceptions[i-1] == 0 and exceptions[i] == 1:
            n01 += 1
        elif exceptions[i-1] == 1 and exceptions[i] == 0:
            n10 += 1
        elif exceptions[i-1] == 1 and exceptions[i] == 1:
            n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        return None, None
    p01 = n01 / (n00 + n01)
    p11 = n11 / (n10 + n11)
    p = (n01 + n11) / (n00 + n01 + n10 + n11)
    if p == 0 or p == 1 or p01 == 0 or p11 == 0:
        return None, None
    lr = -2 * np.log(
        ((1-p)**(n00+n10) * p**(n01+n11)) /
        ((1-p01)**n00 * p01**n01 *
         (1-p11)**n10 * p11**n11))
    p_value = 1 - stats.chi2.cdf(lr, df=1)
    return lr, p_value


print('=' * 65)
print('BACKTESTING RESULTS — Historical VaR at 95%')
print('=' * 65)
print(f"{'Ticker':<10} {'Exceptions':>12} {'Rate':>8} "
      f"{'Kupiec p':>12} {'Christoff p':>13} {'Verdict'}")
print('-' * 65)

for ticker in returns.columns:
    var = var_historical[ticker]
    r = returns[ticker]
    lr_k, pval_k, n_exc, rate = kupiec_test(r, var)
    lr_c, pval_c = christoffersen_test(r, var)
    if pval_k and pval_k > 0.05:
        if pval_c and pval_c > 0.05:
            verdict = 'PASS'
        else:
            verdict = 'CLUSTERING'
    else:
        verdict = 'FAIL'
    print(f'{ticker:<10} {n_exc:>12} {rate*100:>7.2f}% '
          f'{pval_k:>12.4f} {pval_c:>13.4f} {verdict}')

print('\nH0 Kupiec: exception rate = 5% — reject if p-value < 0.05')
print('H0 Christoffersen: exceptions are independent — reject if p-value < 0.05')
print('\nConclusion:')
print('- Kupiec: all assets pass — VaR is correctly calibrated in frequency')
print('- Christoffersen: 4 out of 5 assets fail — exceptions cluster during crises')
print('- Only Nestle passes both tests, confirming its defensive profile')
print('- This clustering justifies the use of dynamic GARCH-based VaR')

## Global Conclusion

This project demonstrates three fundamental insights about market risk measurement:

**1. Static VaR methods are insufficient**  
Parametric, historical and Monte Carlo VaR are correctly calibrated on average (Kupiec test passes for all assets) but fail to capture volatility clustering. When markets enter a crisis, losses concentrate over several consecutive days — something static methods cannot anticipate.

**2. GARCH improves risk measurement significantly**  
Dynamic VaR adapts in real time to market regimes. It expands during crises and contracts during calm periods — giving a much more accurate picture of current risk than a constant volatility estimate.

**3. Machine Learning adds a predictive dimension**  
XGBoost predicts stress days with an AUC-ROC of 0.986. SHAP analysis identifies 5-day cumulative returns and short-term volatility as the strongest early warning signals. This predictive layer complements classical risk measures by anticipating stress before it materializes.

**Final message:**  
No single risk measure is sufficient on its own. VaR gives the threshold, Expected Shortfall gives the severity, GARCH gives the real-time dynamics, and Machine Learning gives the early warning signal. Together they form a complete and robust market risk framework.